In [1]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = (
    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ"
    r"\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist"
    r"\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"
)

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

# Google Cloud credentials
credentials = service_account.Credentials.from_service_account_file(

    r"G:\My Drive\Spacesmith and Wordsmith's Tower\Spacesmith's HQ\Nuclear Energy and Propulsion Engineering\Accenture AI-ML Computational Scientist\Credentials\gen-lang-client-0137385761-b0d89e37e8e5.json"

)

In [3]:
# Create a "Client" object
client = bigquery.Client()

# Construct a reference to the "san_francisco" dataset
dataset_ref = client.dataset("san_francisco", project="bigquery-public-data")

# API request - fetch the dataset
dataset = client.get_dataset(dataset_ref)

# Construct a reference to the "bikeshare_trips" table
table_ref = dataset_ref.table("bikeshare_trips")

# API request - fetch the table
table = client.get_table(table_ref)

# Preview the first five lines of the table
client.list_rows(table, max_results=5).to_dataframe()

,trip_id,duration_sec,start_date,start_station_name,start_station_id,end_date,end_station_name,end_station_id,bike_number,zip_code,subscriber_type
0,1235850,1540,2016-06-11 08:19:00+00:00,San Jose Diridon Caltrain Station,2,2016-06-11 08:45:00+00:00,San Jose Diridon Caltrain Station,2,124,15206,Customer
1,1219337,6324,2016-05-29 12:49:00+00:00,San Jose Diridon Caltrain Station,2,2016-05-29 14:34:00+00:00,San Jose Diridon Caltrain Station,2,174,55416,Customer
2,793762,115572,2015-06-04 09:22:00+00:00,San Jose Diridon Caltrain Station,2,2015-06-05 17:28:00+00:00,San Jose Diridon Caltrain Station,2,190,95391,Customer
3,453845,54120,2014-09-15 16:53:00+00:00,San Jose Diridon Caltrain Station,2,2014-09-16 07:55:00+00:00,San Jose Diridon Caltrain Station,2,127,81,Customer
4,1245113,5018,2016-06-17 20:08:00+00:00,San Jose Diridon Caltrain Station,2,2016-06-17 21:32:00+00:00,San Jose Diridon Caltrain Station,2,153,95070,Customer


In [4]:
# GOAL OF THIS QUERY: 
# 1. Find out how many bike trips happened on every single day in 2015.
# 2. Calculate a "Running Total" (cumulative sum) of trips for the entire year.
#    (For example: If Jan 1st had 10 trips and Jan 2nd had 15, the cumulative total on Jan 2nd is 25)

num_trips_query = """
                  -- Step 1: Create a temporary "helper" table (CTE) named trips_by_day
                  WITH trips_by_day AS
                  (
                      -- Extract just the date (ignoring time) from the start_date
                      SELECT DATE(start_date) AS trip_date,
                          -- Count the total number of trips that happened on this specific date
                          COUNT(*) as num_trips
                      FROM `bigquery-public-data.san_francisco.bikeshare_trips`
                      -- Filter down to only look at trips from the year 2015
                      WHERE EXTRACT(YEAR FROM start_date) = 2015
                      -- Group the results by the trip_date so our COUNT() calculates trips PER DAY
                      GROUP BY trip_date
                  )
                  -- Step 2: Use the helper table to calculate the running total
                  SELECT *,
                      -- Calculate the rolling SUM of the num_trips column
                      SUM(num_trips) 
                          -- The OVER clause is what turns this into an "Analytic Window Function"
                          OVER (
                               -- Sort the rows chronologically by date
                               ORDER BY trip_date
                               -- Define the "window" of rows to add together:
                               -- from the very first row (UNBOUNDED PRECEDING) up to the row we are currently looking at (CURRENT ROW)
                               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
                               ) AS cumulative_trips
                      FROM trips_by_day
                  """

# Run the query, and return a pandas DataFrame
num_trips_result = client.query(num_trips_query).result().to_dataframe(create_bqstorage_client=False)
num_trips_result.head()


,trip_date,num_trips,cumulative_trips
0,2015-01-01,181,181
1,2015-01-02,428,609
2,2015-01-03,283,892
3,2015-01-04,206,1098
4,2015-01-05,1186,2284


In [5]:
# GOAL OF THIS QUERY:
# Track where each individual bike started its day and where it ended its day on October 25, 2015.
# To do this, we use two powerful Analytic Window Functions: FIRST_VALUE() and LAST_VALUE().

start_end_query = """
                  SELECT 
                      bike_number,
                      TIME(start_date) AS trip_time,
                      
                      -- =========================================================================
                      -- ANALYTIC FUNCTION 1: FIRST_VALUE()
                      -- Finds the very first station this specific bike left from today.
                      -- =========================================================================
                      FIRST_VALUE(start_station_id)
                          OVER (
                               -- 1. PARTITION BY breaks the massive table into smaller, separate 
                               --    "partitions" (or groups) based on the bike_number.
                               --    This ensures the calculations are performed separately for EACH bike.
                               PARTITION BY bike_number
                               
                               -- 2. ORDER BY puts the rows within each bike's partition in 
                               --    chronological order (from morning to night).
                               ORDER BY start_date
                               
                               -- 3. ROWS BETWEEN defines the "Window Frame". 
                               --    UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING means for each row,
                               --    its ENTIRE partition (the whole day's trips for that bike) is used.
                               --    This ensures the calculated first station is identical for every row 
                               --    belonging to that specific bike.
                               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
                               ) AS first_station_id,
                               
                      -- =========================================================================
                      -- ANALYTIC FUNCTION 2: LAST_VALUE()
                      -- Finds the very last station this specific bike was returned to today.
                      -- =========================================================================
                      LAST_VALUE(end_station_id)
                          OVER (
                               -- We use the exact same logic here:
                               -- Group by the bike, sort by time, and look at the entire day's trips.
                               PARTITION BY bike_number
                               ORDER BY start_date
                               ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
                               ) AS last_station_id,
                               
                      -- We also grab the start and end station of the current trip for context
                      start_station_id,
                      end_station_id
                      
                  FROM `bigquery-public-data.san_francisco.bikeshare_trips`
                  -- Filter down to just one specific day to keep the data manageable
                  WHERE DATE(start_date) = '2015-10-25' 
                  """

# Run the query, and return a pandas DataFrame
# Note: Added create_bqstorage_client=False to avoid permission issues
start_end_result = client.query(start_end_query).result().to_dataframe(create_bqstorage_client=False)
start_end_result.head()


,bike_number,trip_time,first_station_id,last_station_id,start_station_id,end_station_id
0,22,13:25:00,2,16,2,16
1,25,11:43:00,77,51,77,60
2,25,12:14:00,77,51,60,51
3,29,14:59:00,46,74,46,60
4,29,21:23:00,46,74,60,74
